# Transformation - Joining flight data and Weather data

In [1]:
import sys
from pathlib import Path

project_root = "/notebooks/ML/Flight Delay Prediction Project/ELT"
sys.path.append(str(project_root))

In [2]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pandas as pd

In [3]:
from src.utils.spark_session import get_spark_session
from src.load.flight_data import *
from src.load.weather_data import *

In [4]:
pd.set_option('display.max_columns', None)

In [ ]:
spark = get_spark_session()

### Import flight data

In [6]:
s3_key_fd = "cleaned/Flight_Delay_Prediction_Datasets/flight_data_with_tz/"
fd_df = get_flight_lake_dataset(spark, s3_key_fd)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

In [7]:
fd_df.limit(5).toPandas()

/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,origin_timezone,destination_timezone
0,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,12:05,11:59,-6.0,0.0,0.0,-1,1200-1259,18.0,12:17,14:36,9.0,15:11,14:45,-26.0,0.0,0.0,-2,1500-1559,366.0,319.0,2611.0,11,0.0,0.0,0.0,0.0,0.0,America/New_York,America/Los_Angeles
1,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,15:20,15:38,18.0,18.0,1.0,1,1500-1559,11.0,15:49,23:44,9.0,23:54,23:53,-1.0,0.0,0.0,-1,2300-2359,334.0,295.0,2475.0,10,0.0,0.0,0.0,0.0,0.0,America/Los_Angeles,America/New_York
2,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,18:07,19:01,54.0,54.0,1.0,3,1800-1859,17.0,19:18,21:34,13.0,21:21,21:47,26.0,26.0,1.0,1,2100-2159,374.0,316.0,2611.0,11,6.0,0.0,0.0,0.0,20.0,America/New_York,America/Los_Angeles
3,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,10721,BOS,"Boston, MA",MA,08:35,09:09,34.0,34.0,1.0,2,0800-0859,17.0,09:26,17:46,12.0,17:17,17:58,41.0,41.0,1.0,2,1700-1759,342.0,320.0,2611.0,11,34.0,0.0,7.0,0.0,0.0,America/Los_Angeles,America/New_York
4,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,22:52,22:47,-5.0,0.0,0.0,-1,2200-2259,19.0,23:06,06:57,15.0,07:29,07:12,-17.0,0.0,0.0,-2,0700-0759,337.0,291.0,2475.0,10,0.0,0.0,0.0,0.0,0.0,America/Los_Angeles,America/New_York


## Add UTC timestamps for flight data

In [8]:
from src.transform.flight_data_utc_timestamps import create_utc_timestamps

In [9]:
fd_df_tz = create_utc_timestamps(fd_df)

In [10]:
fd_df_tz.limit(5).toPandas()

/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,origin_timezone,destination_timezone,crs_departure_timestamp,departure_timestamp,wheels_off_timestamp,crs_arrival_timestamp,arrival_timestamp,wheels_on_timestamp,flight_id
0,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,12:05,11:59,-6.0,0.0,0.0,-1,1200-1259,18.0,12:17,14:36,9.0,15:11,14:45,-26.0,0.0,0.0,-2,1500-1559,366.0,319.0,2611.0,11,0.0,0.0,0.0,0.0,0.0,America/New_York,America/Los_Angeles,2025-09-01 16:05:00,2025-09-01 15:59:00,2025-09-01 16:17:00,2025-09-01 22:11:00,2025-09-01 21:45:00,2025-09-01 21:36:00,0
1,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,15:20,15:38,18.0,18.0,1.0,1,1500-1559,11.0,15:49,23:44,9.0,23:54,23:53,-1.0,0.0,0.0,-1,2300-2359,334.0,295.0,2475.0,10,0.0,0.0,0.0,0.0,0.0,America/Los_Angeles,America/New_York,2025-09-01 22:20:00,2025-09-01 22:38:00,2025-09-01 22:49:00,2025-09-02 03:54:00,2025-09-02 03:53:00,2025-09-02 03:44:00,1
2,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,18:07,19:01,54.0,54.0,1.0,3,1800-1859,17.0,19:18,21:34,13.0,21:21,21:47,26.0,26.0,1.0,1,2100-2159,374.0,316.0,2611.0,11,6.0,0.0,0.0,0.0,20.0,America/New_York,America/Los_Angeles,2025-09-01 22:07:00,2025-09-01 23:01:00,2025-09-01 23:18:00,2025-09-02 04:21:00,2025-09-02 04:47:00,2025-09-02 04:34:00,2
3,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,10721,BOS,"Boston, MA",MA,08:35,09:09,34.0,34.0,1.0,2,0800-0859,17.0,09:26,17:46,12.0,17:17,17:58,41.0,41.0,1.0,2,1700-1759,342.0,320.0,2611.0,11,34.0,0.0,7.0,0.0,0.0,America/Los_Angeles,America/New_York,2025-09-01 15:35:00,2025-09-01 16:09:00,2025-09-01 16:26:00,2025-09-01 21:17:00,2025-09-01 21:58:00,2025-09-01 21:46:00,3
4,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,22:52,22:47,-5.0,0.0,0.0,-1,2200-2259,19.0,23:06,06:57,15.0,07:29,07:12,-17.0,0.0,0.0,-2,0700-0759,337.0,291.0,2475.0,10,0.0,0.0,0.0,0.0,0.0,America/Los_Angeles,America/New_York,2025-09-02 05:52:00,2025-09-02 05:47:00,2025-09-02 06:06:00,2025-09-01 11:29:00,2025-09-01 11:12:00,2025-09-01 10:57:00,4


## Perform JOIN with weather data

### Load weather data

In [11]:
s3_key_wd = "staging/Flight_Delay_Prediction_Datasets/weather_data/weather_data.parquet"
wd_df = get_weather_lake_dataset_spark(spark, s3_key_wd)

In [12]:
wd_df.limit(5).toPandas()

/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,station,timestamp,air_temp_f,dew_point_temp_f,relative_humidity,wind_speed_kts,wind_gust_kts,visibility_miles,sky_l1_coverage,sky_l2_coverage,sky_l3_coverage,present_weather_codes,1hr_precipitation_inches,pressure_altimeter_inches
0,HTS,2025-01-01 00:51:00,46.00,37.00,70.64,15.00,25.00,10.00,FEW,BKN,OVC,M,0.01,29.66
1,SAN,2025-01-01 00:51:00,57.00,49.00,74.55,4.00,M,9.00,BKN,M,M,M,0.00,30.05
2,SGF,2025-01-01 00:52:00,36.00,29.00,75.46,11.00,M,10.00,OVC,M,M,M,0.00,30.14
3,BTR,2025-01-01 00:53:00,62.00,39.00,42.56,4.00,M,10.00,CLR,M,M,M,0.00,30.03
4,HOU,2025-01-01 00:53:00,64.00,40.00,41.25,4.00,M,10.00,CLR,M,M,M,0.00,30.07


### JOIN Operation

In [14]:
import pyspark.sql.functions as F

In [17]:
time_window = F.expr("INTERVAL 30 MINUTES")

In [18]:
from src.transform.combine_flight_and_weather_data import join_origin_weather_data, join_destination_weather_data

In [19]:
origin_joined_df = join_destination_weather_data(fd_df_tz, wd_df, time_window)

In [ ]:
origin_joined_df.limit(5).toPandas()

[Stage 12:>                                                        (0 + 2) / 25]